In [1]:
from CNN_NAS.ChildCNNModel import ChildCNNModel
# Some magic so that the notebook will reload the external python script file any time you edit and save the .py file;
%load_ext autoreload
%autoreload 2

([('256', '5', '0'), ('32', '3', '0'), ('32', '5', '1'), ('256', '5', '1'), ('16', '1', '0')], tensor([[-3.9833],
        [-3.8722],
        [-2.9586],
        [-3.3348],
        [-4.1287]], grad_fn=<StackBackward0>))
CNNController(
  (embedding): Embedding(45, 8)
  (rnn): LSTM(8, 32, batch_first=True)
  (fc_filter): Linear(in_features=32, out_features=5, bias=True)
  (fc_kernel): Linear(in_features=32, out_features=3, bias=True)
  (fc_padding): Linear(in_features=32, out_features=3, bias=True)
  (meta_layer): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=False)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=False)
  )
)


In [2]:
import torch
import torch.nn as nn
import time
from torch.utils.data import DataLoader
import os

import utils

import logging
logging.basicConfig(level=logging.INFO, filename=os.path.join(os.getcwd(), 'log.log'), filemode='w')

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

device = utils.get_device_available()
print(torch.__version__)
print(device)

2.4.1
cuda


In [3]:
def is_valid_encoding(encoding):
    if len(encoding) < 3 or encoding[0] != "START" or encoding[-1] != "END":
        return False

    for i in range(1, len(encoding) - 1):
        if not encoding[i].isnumeric():
            return False

    return True


def split_dataset(data, labels, split_ratio=0.8):
    dataset = torch.utils.data.TensorDataset(data, labels)
    train_size = int(split_ratio * len(dataset))
    test_size = len(dataset) - train_size

    train_set, test_set = torch.utils.data.random_split(dataset, [train_size, test_size])

    train_data, train_labels = zip(*train_set)
    train_data = torch.stack(train_data)
    train_labels = torch.stack(train_labels)

    test_data, test_labels = zip(*test_set)
    test_data = torch.stack(test_data)
    test_labels = torch.stack(test_labels)

    return (train_data, train_labels), (test_data, test_labels)

## CIFAR-100

In [4]:
dataset="cifar100"

data_path = utils.check_cifar_dataset_exists()

dataset_train_data,dataset_train_label = (torch.load(data_path + f'{dataset}/train_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/train_label.pt', weights_only=True))

dataset_test_data,dataset_test_label = (torch.load(data_path + f'{dataset}/test_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/test_label.pt', weights_only=True))


num_channels = 1

if len(dataset_train_data.size())==4:
    num_channels=dataset_train_data.size(1)



num_classes = dataset_train_label.unique().size(0)
height = dataset_train_data.size(-2)
width = dataset_train_data.size(-1)

print(f"Height: {height}")
print(f"Width: {width}")
print(f"Number of channels: {num_channels}")
print(f"Number of classes:  {num_classes}")



Height: 32
Width: 32
Number of channels: 3
Number of classes:  100


## Load  predefined model encoding

In [5]:
import predefined_models
# Defined by data

base_model_encoding_dict = {}

base_model_encoding_dict["Benchmark_Model"] = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["Lenet"] = predefined_models.get_lenet(input_channels=num_channels, output_dim=num_classes)
base_model_encoding_dict["VGG_11"] = predefined_models.get_vgg11(input_channels=num_channels, output_dim=num_classes)
# base_model_encoding_dict["Alexnet"] = predefined_models.get_alexnet(input_channels=num_channels, output_dim=num_classes)
# 

# base_model = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)

# print(base_model)

In [6]:
for base_model in base_model_encoding_dict:
    print(base_model)
    print(ChildCNNModel(base_model_encoding_dict[base_model], num_channels,height,width, num_classes))
    

Benchmark_Model
ChildCNNModel(
  (model): Sequential(
    (0): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Flatten(start_dim=1, end_dim=-1)
    (10): Linear(in_features=512, out_features=512, bias=True)
    (11): ReLU()
    (12): Linear(in_features=512, out_features=256, bias=True)
    (13): ReLU()
    (14): Linear(in_features=256, out_features=100, bias=True)
  )
)
Lenet
ChildCNNModel(
  (model): Sequential(
    (0): Conv2d(3, 50, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPoo

In [8]:
from CNN_NAS.ChildCNNModel import ChildCNNModel

#Load and run
logger.info("###############################################")
logger.info("STARTING TRAINING")
logger.info("###############################################")


# print(model)
utils.cleanup_child_model()
print("Done cleaning")
for base_model in base_model_encoding_dict:
    total_epochs=10
    name = base_model
    for i in range(3):
        model = ChildCNNModel(base_model_encoding_dict[base_model], num_channels,height,width, num_classes).to(device)
        loss,train_time = model.train_model(data=dataset_train_data,label=dataset_train_label,epochs=total_epochs,dataset_name=dataset)
        test_accuracy = model.evaluate_model(data=dataset_test_data,labels=dataset_test_label,dataset_name=dataset)
        print(f"Model {name} Train loss:{loss} Test Accuracy: {test_accuracy} with total epochs {total_epochs} in dataset {dataset} for time {train_time}")
        total_epochs += 10
        utils.cleanup_child_model(model)

Done cleaning
Model Benchmark_Model Train loss:2.1551268830299377 Test Accuracy: (0.443, 2.113781168460846) with total epochs 10 in dataset cifar100 for time 40.879830837249756


KeyboardInterrupt: 

## CIFAR

In [10]:
def benchmark_dataset(dataset_name):
    dataset=dataset_name
    
    if dataset== "cifar100":
        data_path = utils.check_cifar_dataset_exists()
    elif dataset== "cifar":
        data_path = utils.check_cifar_dataset_exists()
    elif dataset== "mnist":
        data_path = utils.check_mnist_dataset_exists()
    
    dataset_train_data,dataset_train_label = (torch.load(data_path + f'{dataset}/train_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/train_label.pt', weights_only=True))
    
    dataset_test_data,dataset_test_label = (torch.load(data_path + f'{dataset}/test_data.pt', weights_only=True), torch.load(data_path + f'{dataset}/test_label.pt', weights_only=True))
    
    
    num_channels = 1
    
    if len(dataset_train_data.size())==4:
        num_channels=dataset_train_data.size(1)
    
    
    num_classes = dataset_train_label.unique().size(0)
    height = dataset_train_data.size(-2)
    width = dataset_train_data.size(-1)
    
    print(f"Height: {height}")
    print(f"Width: {width}")
    print(f"Number of channels: {num_channels}")
    print(f"Number of classes:  {num_classes}")
    
    import predefined_models
    base_model_encoding_dict = {}
    
    base_model_encoding_dict["Benchmark_Model"] = predefined_models.get_benchmarkModel(input_channels=num_channels, output_dim=num_classes)
    base_model_encoding_dict["Lenet"] = predefined_models.get_lenet(input_channels=num_channels, output_dim=num_classes)
    base_model_encoding_dict["VGG_11"] = predefined_models.get_vgg11(input_channels=num_channels, output_dim=num_classes)
    
    from CNN_NAS.ChildCNNModel import ChildCNNModel

    #Load and run
    logger.info("###############################################")
    logger.info("STARTING TRAINING")
    logger.info("###############################################")
    
    
    # print(model)
    utils.cleanup_child_model()
    print("Done cleaning")
    for base_model in base_model_encoding_dict:
        total_epochs=10
        name = base_model
        for i in range(3):
            model = ChildCNNModel(base_model_encoding_dict[base_model], num_channels,height,width, num_classes).to(device)
            loss,train_time = model.train_model(data=dataset_train_data,label=dataset_train_label,epochs=total_epochs,dataset_name=dataset)
            test_accuracy = model.evaluate_model(data=dataset_test_data,labels=dataset_test_label,dataset_name=dataset)
            print(f"Model {name} Train loss:{loss} Test Accuracy: {test_accuracy} with total epochs {total_epochs} in dataset {dataset} for time {train_time}")
            total_epochs += 10
            utils.cleanup_child_model(model)



In [12]:
benchmark_dataset("cifar")

Height: 32
Width: 32
Number of channels: 3
Number of classes:  10
Done cleaning
Model Benchmark_Model Train loss:0.6715180802345276 Test Accuracy: (0.7779, 0.6355839329957962) with total epochs 10 in dataset cifar for time 40.87065649032593


KeyboardInterrupt: 